In [1]:
%pip install pymupdf pillow easyocr
%pip install torch torchvision torchaudio

  Using cached pymupdf-1.27.2.2-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (3.4 kB)
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
  Using cached python_bidi-0.6.7-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
  Using cached shapely-2.1.2-cp312-cp


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Group names: Chi Kien, Kedar Hegde, Isaiah Austin, Alan Chavarin

# 1. Data Loading & Preprocessing
Brief: This section loads NDA documents and preprocesses them into clauses.

In [4]:
import fitz  # PyMuPDF
import io
import numpy as np
import easyocr
from PIL import Image
import torch


# Initialize OCR reader once per session (downloads model files on first run)
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())


def extract_text_from_pdf(file_path, zoom=2.0):
    """Extract embedded PDF text and fallback to EasyOCR for image-only pages."""
    doc = fitz.open(file_path)
    text_parts = []

    for page in doc:
        embedded_text = page.get_text().strip()
        if embedded_text:
            text_parts.append(embedded_text)
            continue

        # OCR fallback for scanned/image-only pages
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_np = np.array(img)

        # detail=0 returns text strings only, paragraph=True merges nearby lines
        ocr_lines = ocr_reader.readtext(img_np, detail=0, paragraph=True)
        ocr_text = "\n".join(line.strip() for line in ocr_lines if line and line.strip())
        text_parts.append(ocr_text)

    return "\n".join(part for part in text_parts if part)


# Load files
nda1_text = extract_text_from_pdf("./data-original/NDA-1.pdf")
nda2_text = extract_text_from_pdf("./data-altered/NDA-1_ALT.pdf")

# Preview
#print(nda1_text[:1000])
print(nda2_text[:1000])

Classroom Use of Confidential Information 
Non-Disclosure Agreement (NDA) Template Package 
 
for Course Instructors 
Do not use the attached template if students 
are receiving third party proprietary 
confidential information: 
 ...as part of a student placement, internship, 
co-curricular or other off-site activity. 
Contact the Office of the Vice-Provost, 
Students for applicable templates and 
additional information related to these 
activities; or 
 ...to conduct research under the supervision 
of a principal investigator. Contact the 
Innovations & Partnerships Office for 
applicable templates and additional 
information related to supervised research. 
 
Use the attached template if: 
□ You have determined that receiving 
confidential information from a third party 
for classroom use would be highly beneficial 
to your students enrolled in the academic 
course you are teaching; 
□ You have obtained approval from your 
academic unit/division head, for you and 
your students to r

In [5]:
import re

def clean_text(text):
    # remove weird line breaks
    text = text.replace("\n", " ")
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    # remove weird symbols
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    
    return text.strip()

nda1_clean = clean_text(nda1_text)
nda2_clean = clean_text(nda2_text)

print(nda1_clean[:1000])
print(nda2_clean[:1000])

Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your students to receive confidential information from

In [6]:
def split_into_clauses(text, min_len=50):
    """Split text into clause-like chunks with OCR-friendly fallbacks."""
    cleaned = text.strip()
    if not cleaned:
        return []

    # Insert split hints for common section/list markers that OCR often flattens.
    normalized = re.sub(r"\s+(?=(?:\d+\.|[a-z]\)|Instructions:|APPENDIX\s+[A-Z]|Page\s+\d+\s+of\s+\d+))", "\n", cleaned)

    # First pass: split by explicit markers.
    chunks = re.split(r"\n+", normalized)
    chunks = [c.strip() for c in chunks if len(c.strip()) > min_len]

    # If still too coarse, fallback to sentence-group chunking.
    if len(chunks) <= 6:
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z\[])", cleaned)
        sentences = [s.strip() for s in sentences if s.strip()]

        grouped = []
        current = []
        current_len = 0
        for sentence in sentences:
            current.append(sentence)
            current_len += len(sentence)
            if current_len >= 170:
                grouped.append(" ".join(current))
                current = []
                current_len = 0
        if current:
            grouped.append(" ".join(current))

        fallback = [c.strip() for c in grouped if len(c.strip()) > min_len]
        if len(fallback) > len(chunks):
            return fallback

    return chunks

nda1_clauses = split_into_clauses(nda1_clean)
nda2_clauses = split_into_clauses(nda2_clean)

print("Number of clauses(1):", len(nda1_clauses))
print("\nExample clause:\n", nda1_clauses[0])

print("Number of clauses(2):", len(nda2_clauses))
print("\nExample clause:\n", nda2_clauses[0])

Number of clauses(1): 29

Example clause:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your studen

In [7]:
%pip uninstall -y transformers
%pip install "transformers[sentencepiece]" --upgrade --force-reinstall

Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached sentencepiece-0

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\alanc\\AppData\\Roaming\\Python\\Python313\\site-packages\\transformers\\models\\metaclip_2\\__init__.py'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.11.0+cu130
GPU available: True


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_text(text):
    input_text = "summarize: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=20
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 131/131 [00:03<00:00, 43.14it/s]


In [10]:
sample_clause = nda1_clauses[1][:500]

print("ORIGINAL:\n", sample_clause)
print("\nSUMMARY:\n", summarize_text(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

SUMMARY:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.


# 2. Feature 1: Summarization

## Baseline
Uses T5-small model for basic summarization.

## Improved
Uses prompt engineering and beam search to improve output quality.

## Comparison
Outputs from both approaches are compared.

In [11]:
def summarize_text_improved(text):
    input_text = "summarize key points: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=25,
        num_beams=4,           # better quality
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
print("BASELINE:\n", summarize_text(sample_clause))
print("\nIMPROVED:\n", summarize_text_improved(sample_clause))

BASELINE:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.

IMPROVED:
 the academic unit/division head with signing authority is identified. the academic unit/division head is not typically authorized to execute an NDA.


# 3. Feature 2: Risk Detection

## Baseline (Zero-shot)
Uses a pre-trained model to classify clauses as high or low risk.

## Improved (Logistic Regression)
Uses a trained machine learning model on labeled data.

## Comparison
Predictions from both models are compared.

In [13]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:28<00:00, 18.35it/s]


In [14]:
labels = ["high risk", "low risk"]

result = classifier(sample_clause, candidate_labels=labels)

print(result)

{'sequence': '1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.', 'labels': ['high risk', 'low risk'], 'scores': [0.8147278428077698, 0.18527214229106903]}


In [15]:
data = [
    ("Confidential information must not be disclosed to third parties", "high risk"),
    ("Recipient must protect confidential data", "high risk"),
    ("All confidential information must be returned after termination", "high risk"),
    ("This agreement lasts for two years", "low risk"),
    ("The agreement is governed by local law", "low risk"),
    ("Students may use the information only for coursework", "low risk"),
]

texts = [x[0] for x in data]
labels = [x[1] for x in data]

In [16]:
%pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [18]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X, labels)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [19]:
test_X = vectorizer.transform([sample_clause])
prediction = model_lr.predict(test_X)

print("Prediction:", prediction[0])

Prediction: low risk


In [20]:
for clause in nda1_clauses[:5]:
    print("\nCLAUSE:\n", clause[:200])
    
    # Zero-shot
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    zero_shot_pred = result["labels"][0]
    
    # Logistic regression
    test_X = vectorizer.transform([clause])
    lr_pred = model_lr.predict(test_X)[0]
    
    print("Zero-shot:", zero_shot_pred)
    print("Logistic Regression:", lr_pred)


CLAUSE:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary con
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are 
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 2. Customize and complete all areas noted with an "INSERT...", which require content to be inserted by University of Toronto faculty or staff, as appopriate.
Zero-shot: high risk
Logistic Regression: high risk

CLAUSE:
 3. Notify the authorized signatory if changes have been made to this template by the third party prior to execution because proposed amendments require legal review.
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 4. Once the NDA is 

# 4. Feature 3: Jargon Explanation

## Baseline
Simple prompt-based explanation.

## Improved
Enhanced prompt and decoding for better clarity.

## Comparison
Outputs are compared to evaluate improvement.

In [21]:
def explain_clause_v1(text):
    input_text = "explain in simple terms: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [22]:
print("ORIGINAL:\n", sample_clause)
print("\nBASELINE EXPLANATION:\n", explain_clause_v1(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

BASELINE EXPLANATION:
 : 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.


In [23]:
def explain_clause_v2(text):
    input_text = "rewrite this legal sentence in very simple plain English for a non-lawyer: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40,
        num_beams=5,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [24]:
print("\nIMPROVED EXPLANATION:\n", explain_clause_v2(sample_clause))


IMPROVED EXPLANATION:
 1. Consult your academic divisions guidelines regarding the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are typically not typically authorized to execute an NDA.


# 5. Feature 4: Version Comparison and New Risk Detection

## foal
Compare two versions of an NDA, highlight clause level changes, and flag newly introduced risks

## Approach
- match clauses across versions with TF-IDF cosine similarity
- categorize changes as added, removed, or modified
- run risk classification on changed clauses and identify risk increases
- print a compact review report for legal/compliance triage

In [25]:
from difflib import unified_diff
from sklearn.metrics.pairwise import cosine_similarity


def predict_risk_label(clause):
    """Use the existing zero-shot classifier for stable risk labels."""
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    return result["labels"][0], float(result["scores"][0])


def compare_clause_sets(
    old_clauses,
    new_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
):
    """
    Match clauses one-to-one using a broader candidate pool.
    Returns: added, removed, modified, unchanged, structural_changes, diagnostic
    """
    if not old_clauses and not new_clauses:
        return [], [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not old_clauses:
        return new_clauses, [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not new_clauses:
        return [], old_clauses, [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}

    vec = TfidfVectorizer(ngram_range=(1, 2))
    old_X = vec.fit_transform(old_clauses)
    new_X = vec.transform(new_clauses)

    sim = cosine_similarity(new_X, old_X) 

    candidate_pairs = []
    for new_idx in range(sim.shape[0]):
        row = sim[new_idx]
        best_old_idx = int(row.argmax())
        best_score = float(row[best_old_idx])
        candidate_pairs.append((best_score, new_idx, best_old_idx))

        for old_idx, score in enumerate(row):
            score = float(score)
            if score >= candidate_floor and old_idx != best_old_idx:
                candidate_pairs.append((score, new_idx, old_idx))

    candidate_pairs.sort(reverse=True, key=lambda x: x[0])

    matched_new = set()
    matched_old = set()
    modified = []
    unchanged = []

    for score, new_idx, old_idx in candidate_pairs:
        if new_idx in matched_new or old_idx in matched_old:
            continue
        if score < modified_threshold:
            continue

        matched_new.add(new_idx)
        matched_old.add(old_idx)

        old_clause = old_clauses[old_idx]
        new_clause = new_clauses[new_idx]
        if old_clause.strip() == new_clause.strip():
            unchanged.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })
        else:
            modified.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })

    unmatched_new_idx = [i for i in range(len(new_clauses)) if i not in matched_new]
    unmatched_old_idx = [i for i in range(len(old_clauses)) if i not in matched_old]

    structural_changes = []
    structural_new = set()
    structural_old = set()

    for new_idx in unmatched_new_idx:
        links = []
        for old_idx in unmatched_old_idx:
            score = float(sim[new_idx, old_idx])
            if score >= structural_threshold:
                links.append({"old_idx": old_idx, "similarity": score})

        if links:
            links = sorted(links, key=lambda x: x["similarity"], reverse=True)[:3]
            structural_changes.append(
                {
                    "new_clause": new_clauses[new_idx],
                    "linked_old_clauses": [
                        {"old_clause": old_clauses[item["old_idx"]], "similarity": item["similarity"]}
                        for item in links
                    ],
                }
            )
            structural_new.add(new_idx)
            for item in links:
                structural_old.add(item["old_idx"])

    added = [new_clauses[i] for i in unmatched_new_idx if i not in structural_new]
    removed = [old_clauses[i] for i in unmatched_old_idx if i not in structural_old]

    diagnostics = {
        "matched_pairs": len(matched_new),
        "candidate_pairs": len(candidate_pairs),
        "threshold": modified_threshold,
        "candidate_floor": candidate_floor,
        "structural_threshold": structural_threshold,
    }

    return added, removed, modified, unchanged, structural_changes, diagnostics


def clause_word_diff(old_clause, new_clause, context_lines=1):
    old_words = old_clause.split()
    new_words = new_clause.split()
    diff_lines = list(unified_diff(old_words, new_words, fromfile="old", tofile="new", lineterm="", n=context_lines))
    return "\n".join(diff_lines)

In [26]:
def analyze_version_changes(
    old_clauses,
    new_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
):
    added, removed, modified, unchanged, structural_changes, diagnostics = compare_clause_sets(
        old_clauses,
        new_clauses,
        modified_threshold=modified_threshold,
        candidate_floor=candidate_floor,
        structural_threshold=structural_threshold,
    )

    added_risks = []
    for clause in added:
        label, score = predict_risk_label(clause)
        if label == "high risk":
            added_risks.append({
                "clause": clause,
                "risk_label": label,
                "risk_score": score,
            })

    modified_risk_deltas = []
    for item in modified:
        old_label, old_score = predict_risk_label(item["old_clause"])
        new_label, new_score = predict_risk_label(item["new_clause"])

        is_increase = (old_label == "low risk" and new_label == "high risk") or (
            old_label == "high risk" and new_label == "high risk" and new_score > old_score
        )

        if is_increase:
            modified_risk_deltas.append({
                "old_clause": item["old_clause"],
                "new_clause": item["new_clause"],
                "similarity": item["similarity"],
                "old_risk_label": old_label,
                "old_risk_score": old_score,
                "new_risk_label": new_label,
                "new_risk_score": new_score,
                "diff": clause_word_diff(item["old_clause"], item["new_clause"]),
            })

    return {
        "added": added,
        "removed": removed,
        "modified": modified,
        "unchanged": unchanged,
        "structural_changes": structural_changes,
        "added_high_risk": added_risks,
        "modified_risk_increase": modified_risk_deltas,
        "diagnostics": diagnostics,
    }

In [27]:
# print("nda1_clauses", nda1_clauses)
# print("nda2_clauses", nda2_clauses)

# Compare NDA-1 (old) vs NDA 2 (new)
version_report = analyze_version_changes(
    nda1_clauses,
    nda2_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
)

print("=== VERSION CHANGE SUMMARY ===")
print("Added clauses:", len(version_report["added"]))
print("Removed clauses:", len(version_report["removed"]))
print("Modified clauses:", len(version_report["modified"]))
print("Unchanged matched clauses:", len(version_report["unchanged"]))
print("Structural merge/split changes:", len(version_report["structural_changes"]))
print()
print("Newly added high-risk clauses:", len(version_report["added_high_risk"]))
print("Modified clauses with risk increase:", len(version_report["modified_risk_increase"]))

print("\n=== MATCHING DIAGNOSTICS ===")
print(version_report["diagnostics"])

print("\n=== ADDED HIGH-RISK CLAUSES (sample) ===")
for i, item in enumerate(version_report["added_high_risk"][:3], 1):
    print(f"[{i}] score={item['risk_score']:.3f}")
    print(item["clause"][:300], "\n")

print("\n=== STRUCTURAL CHANGES (sample) ===")
for i, item in enumerate(version_report["structural_changes"][:3], 1):
    print(f"[{i}] new clause:", item["new_clause"][:220])
    for j, linked in enumerate(item["linked_old_clauses"], 1):
        print(f"   linked old {j} sim={linked['similarity']:.3f}:", linked["old_clause"][:180])
    print()

print("\n=== MODIFIED CLAUSES WITH RISK INCREASE (sample) ===")
for i, item in enumerate(version_report["modified_risk_increase"][:3], 1):
    print(f"[{i}] similarity={item['similarity']:.3f}")
    print(f"old: {item['old_risk_label']} ({item['old_risk_score']:.3f})")
    print(f"new: {item['new_risk_label']} ({item['new_risk_score']:.3f})")
    print("old clause:", item["old_clause"][:220])
    print("new clause:", item["new_clause"][:220])
    print("word diff:")
    print(item["diff"][:1200])
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== VERSION CHANGE SUMMARY ===
Added clauses: 3
Removed clauses: 13
Modified clauses: 8
Unchanged matched clauses: 8
Structural merge/split changes: 0

Newly added high-risk clauses: 3
Modified clauses with risk increase: 2

=== MATCHING DIAGNOSTICS ===
{'matched_pairs': 16, 'candidate_pairs': 30, 'threshold': 0.58, 'candidate_floor': 0.35, 'structural_threshold': 0.42}

=== ADDED HIGH-RISK CLAUSES (sample) ===
[1] score=0.800
d) with anyone, including other students, outside of the course. 

[2] score=0.708
11. In the event of breach, Discloser may seek injunctive relief in addition to any other remedies available at law or in equity. 

[3] score=0.684
12. UT will ensure that access to Confidential Information is restricted to Students and instructional personnel with a need to know, and that such access is revoked at course completion. This Agreement is made effective on the Effective Date. By:______________________________________ Title:________ 


=== STRUCTURAL CHANGES (sample) ==